''''''''''''''''''''''''''''''''''''''''''''''''''
#Author: Shane Tomb
#Date Created 10/27/2026
#Last Updated 11/21/2026
#Version 1.25 (new file)

# Blender Guide Agents
A multi-agent system that writes step-by-step Blender guides and tailors them to your .blend file.

**Setup:** add a Hugging Face token (with Inference permissions) in Colab's left sidebar under 🔑 **Secrets** named HF_TOKEN, and enable notebook access. Then run all cells.

**Agents:** Project Analyst -> Planner -> Guide Writer -> Reviewer

**Variables:**

-HF_TOKEN- Stores huggingface token from Hugging Face account, accesses the 🔑 menu of Colab's left sidebar.

-DEFAULT_MODEL- Defines HF (Hugging Face) model used by the agent.

-BLENDER_BIN- File container for Blender Files within the agent.

-INSPECT_SCRIPT- Contains a script to be sent into the "headless" Blender to get metadata from the .blend file format.

-out- Metadata obtained from INSPECT_SCRIPT after import into .blend file.

-Client- Initializes connection with Hugging Face model.

-resp- Sets up client with information.

-messages- sets up roles for the agents as described below.
  
    -ANALYST- Analyzes the .blend file to make observations to be used to make a guide of next steps.
    -PLANNER- Outlines the steps in simplest terms to be used for creating the full guide.
    -WRITER- Works with the outline given to give exact constructions including mistakes to avoid to give as much relevant info. as possible to the enduser.
    -REVIEWER- Works with the rest of the given output and corrects any uncertainties including those with inaccurate shortcuts, filepaths, misspellings, etc.
-Trace, Analysis- Setting up for gradio UI.

-Status- Status of Agents working on the output.

-LOCAL_MODEL- Locally stored hugging face model, replaces DEFAULT_MODEL, works as fail-safe.

-tok- Tokenizer for LOCAL_MODEL

''''''''''''''''''''''''''''''''''''''''''''''''''

In [1]:
# 1. Install dependencies + headless Blender (used to read .blend files).
!pip install -q gradio huggingface_hub
!apt-get -qq install -y libxi6 libxxf86vm1 libxfixes3 libxrender1 libgl1 libxkbcommon0 libsm6 > /dev/null
!if [ ! -d /content/blender ]; then wget -q https://download.blender.org/release/Blender4.2/blender-4.2.3-linux-x64.tar.xz -O /content/blender.tar.xz && tar -xf /content/blender.tar.xz -C /content && mv /content/blender-4.2.3-linux-x64 /content/blender; fi
!/content/blender/blender --version | head -1

Blender 4.2.3 LTS


In [2]:
# 2. Config, importing user keys, huggingface client and token, model, and Blender file repo.
import os, json, subprocess
from google.colab import userdata
from huggingface_hub import InferenceClient

HF_TOKEN = userdata.get("HF_TOKEN")
DEFAULT_MODEL = "Qwen/Qwen2.5-72B-Instruct"   # Works for any chat model available on HF.
BLENDER_BIN = "/content/blender/blender"

In [3]:
# 3. Script that runs INSIDE headless Blender and dumps a summary of the project
INSPECT_SCRIPT = r"""
import bpy, json, sys
out = sys.argv[sys.argv.index("--") + 1]
r = lambda v: [round(x, 3) for x in v]
d = {"saved_with_blender": ".".join(map(str, bpy.data.version))}

d["scenes"] = []
for s in bpy.data.scenes:
    d["scenes"].append({
        "name": s.name, "render_engine": s.render.engine,
        "resolution": [s.render.resolution_x, s.render.resolution_y],
        "frame_range": [s.frame_start, s.frame_end], "fps": s.render.fps,
        "camera": s.camera.name if s.camera else None,
        "unit_system": s.unit_settings.system,
    })

d["objects"] = []
for o in list(bpy.data.objects)[:150]:
    e = {"name": o.name, "type": o.type, "location": r(o.location),
         "rotation_deg": [round(x * 57.2958, 1) for x in o.rotation_euler],
         "scale": r(o.scale), "dimensions": r(o.dimensions),
         "parent": o.parent.name if o.parent else None,
         "collections": [c.name for c in o.users_collection],
         "modifiers": [{"name": m.name, "type": m.type} for m in o.modifiers],
         "materials": [s.material.name for s in o.material_slots if s.material],
         "has_animation": bool(o.animation_data and o.animation_data.action)}
    if o.type == "MESH":
        e["vertices"] = len(o.data.vertices); e["faces"] = len(o.data.polygons)
        e["uv_maps"] = [u.name for u in o.data.uv_layers]
        e["vertex_groups"] = [v.name for v in o.vertex_groups]
        e["shape_keys"] = bool(o.data.shape_keys)
    if o.type == "LIGHT": e["light_type"] = o.data.type; e["energy"] = o.data.energy
    if o.type == "ARMATURE": e["bones"] = len(o.data.bones)
    d["objects"].append(e)

d["materials"] = []
for m in bpy.data.materials:
    nodes = [n.bl_idname.replace("ShaderNode", "") for n in m.node_tree.nodes] if m.use_nodes and m.node_tree else []
    d["materials"].append({"name": m.name, "nodes": nodes})

d["collections"] = [c.name for c in bpy.data.collections]
d["images"] = [i.name for i in bpy.data.images if i.name != "Render Result"]
d["node_groups"] = [n.name for n in bpy.data.node_groups]
d["actions"] = [a.name for a in bpy.data.actions]
d["world"] = bpy.data.worlds[0].name if bpy.data.worlds else None
json.dump(d, open(out, "w"), indent=1)
"""
open("/content/inspect_blend.py", "w").write(INSPECT_SCRIPT)
# Handles the file access and dumping of file summary.
def inspect_blend(path):
    out = "/content/blend_summary.json"
    if os.path.exists(out): os.remove(out)
    r = subprocess.run([BLENDER_BIN, "-b", path, "--python", "/content/inspect_blend.py", "--", out],
                       capture_output=True, text=True, timeout=180)
    if not os.path.exists(out):
        raise RuntimeError("Could not read .blend:\n" + (r.stderr or r.stdout)[-1500:])
    return json.load(open(out))

In [4]:
# 4. Initializes agents, defines their roles, and defines status messages.
def llm(system, user, model, max_tokens=2500, temperature=0.4):
    client = InferenceClient(model=model, token=HF_TOKEN)
    resp = client.chat_completion(
        messages=[{"role": "system", "content": system}, {"role": "user", "content": user}],
        max_tokens=max_tokens, temperature=temperature)
    return resp.choices[0].message.content

ANALYST = """You are a Blender project analyst. You receive a JSON summary of a .blend file and a user's goal.
List concrete observations relevant to the goal: what already exists, what is missing, likely problems
(e.g. unapplied scale, no UVs, heavy poly counts, missing materials, no camera/lights, wrong render engine),
and which existing object names the guide should reference. Be factual: only state what the JSON supports.
Output a short bullet list."""

PLANNER = """You are a Blender curriculum planner. Given the user's goal, skill level, Blender version and
(optional) project analysis, produce a numbered outline of 6-15 steps in a sensible workflow order.
Each line: step title + one sentence on what it achieves. Do not write the full guide."""

WRITER = """You are an expert Blender instructor writing a step-by-step guide in Markdown.
Follow the outline. For each step give: the exact menu path or editor/panel, keyboard shortcuts,
specific values where helpful, and a short 'Why' note. Add a 'Common mistakes' line where relevant.
Adjust depth to the skill level. If a project analysis is provided, reference real object, material and
collection names from the file and call out issues found in it. Use the stated Blender version's UI.
If you are not sure a menu path or shortcut exists in that version, say so rather than inventing it."""

REVIEWER = """You are a meticulous Blender technical reviewer. Review the draft guide against the outline,
the user's goal, the stated Blender version, and the project analysis. Fix wrong or invented shortcuts,
menu paths and operator names, reorder steps that are in the wrong order, and remove references to objects
that are not in the project analysis. Return ONLY the corrected final guide in Markdown, then a section
'## Notes on accuracy' listing anything you changed or were unsure about."""

def run_pipeline(prompt, blend_file, skill, version, model):
    trace, analysis = [], "No .blend file provided."
    status = lambda s, g="", t=None: (s, g, "\n\n".join(trace) if t is None else t)

    if blend_file:
        path = getattr(blend_file, "name", blend_file)
        yield status("🔍 Reading .blend file in headless Blender...")
        summary = inspect_blend(path)
        raw = json.dumps(summary)[:9000]
        trace.append("### Raw project summary\n```json\n" + raw[:3000] + "\n```")
        yield status("🧠 Project Analyst working...")
        analysis = llm(ANALYST, f"Goal: {prompt}\n\nProject JSON:\n{raw}", model, 900, 0.2)
        trace.append("### Project Analyst\n" + analysis)

    ctx = f"Goal: {prompt}\nSkill level: {skill}\nBlender version: {version}\n\nProject analysis:\n{analysis}"
    yield status("🗺️ Planner working...")
    outline = llm(PLANNER, ctx, model, 900, 0.3)
    trace.append("### Planner outline\n" + outline)

    yield status("✍️ Guide Writer working...")
    draft = llm(WRITER, f"{ctx}\n\nOutline:\n{outline}", model, 3000, 0.5)
    trace.append("### Writer draft\n" + draft)

    yield status("🔎 Reviewer checking accuracy...", draft)
    final = llm(REVIEWER, f"{ctx}\n\nOutline:\n{outline}\n\nDraft:\n{draft}", model, 3500, 0.2)
    yield status("✅ Done", final)

In [8]:
# 5. Sets up and runs the UI in a window/local ip site.
import gradio as gr

def ui_fn(prompt, blend_file, skill, version, model):
    if not prompt.strip():
        yield "⚠️ Please enter a prompt.", "", ""
        return
    try:
        yield from run_pipeline(prompt, blend_file, skill, version, model or DEFAULT_MODEL)
    except Exception as e:
        yield f"❌ {type(e).__name__}", f"```\n{e}\n```", ""

with gr.Blocks(title="Blender Guide Agents") as demo:
    gr.Markdown("# 🧊 Blender Guide Agents\nDescribe what you want to make or learn, optionally upload a `.blend` file for project-specific tips.")
    with gr.Row():
        with gr.Column(scale=1):
            prompt = gr.Textbox(label="What do you want a guide for?", lines=4,
                                placeholder="e.g. How do I UV unwrap and texture the character in my file?")
            blend = gr.File(label="Optional: .blend file", file_types=[".blend"])
            skill = gr.Dropdown(["Beginner", "Intermediate", "Advanced"], value="Beginner", label="Skill level")
            version = gr.Dropdown(["4.2 LTS", "4.1", "4.0", "3.6 LTS", "2.93 LTS"], value="4.2 LTS",
                                  label="Your Blender version", allow_custom_value=True)
            model = gr.Textbox(label="Hugging Face model ID", value=DEFAULT_MODEL)
            go = gr.Button("Generate guide", variant="primary")
        with gr.Column(scale=2):
            status = gr.Markdown()
            guide = gr.Markdown(label="Guide")
            with gr.Accordion("Agent trace (analysis, outline, draft)", open=False):
                trace = gr.Markdown()
    go.click(ui_fn, [prompt, blend, skill, version, model], [status, guide, trace])

demo.queue().launch(share=True, debug=False)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://5d33c18e81467eef19.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [5]:
#Failsafe to allow for hugging face to work regardless of pro or free user account

!pip install -q -U transformers accelerate bitsandbytes
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

LOCAL_MODEL = "Qwen/Qwen2.5-7B-Instruct"   # use "Qwen/Qwen2.5-3B-Instruct" if memory or speed is a problem
tok = AutoTokenizer.from_pretrained(LOCAL_MODEL)
lm = AutoModelForCausalLM.from_pretrained(
    LOCAL_MODEL, device_map="auto",
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16))

def llm(system, user, model=None, max_tokens=2500, temperature=0.4):
    msgs = [{"role": "system", "content": system}, {"role": "user", "content": user}]
    text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tok(text, return_tensors="pt").to(lm.device)
    with torch.no_grad():
        out = lm.generate(**inputs, max_new_tokens=max_tokens, do_sample=True,
                          temperature=temperature, top_p=0.9)
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/27.8k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

In [7]:
#failsafe version of app if other does not work
try:
    demo.close()
except Exception:
    pass

demo.queue().launch(share=False, debug=False, prevent_thread_lock=True)

from google.colab import output
output.serve_kernel_port_as_window(7860)   # opens the app in a new tab; click the link it prints
# or, to embed it inline under the cell instead:
# output.serve_kernel_port_as_iframe(7860, height=900)

NameError: name 'demo' is not defined